# TwinLiteNet+ 자체 트랙 파인튜닝 (Colab)

UMK `track_drive` 패키지가 쓰는 TwinLiteNet(da/ll 듀얼헤드) 대신 **TwinLiteNet+**를
우리 트랙 데이터로 파인튜닝해서 `.onnx`로 내보내는 노트북. 배경: 실차 영상 분석 결과
da/`ll` 세그멘테이션이 (1) 노란선이 거의 항상 안 잡히고 (2) 역광/글레어에서 완전히
미검출되는 문제가 확인됐고(레포 README `track_drive/track_drive/README.md` §2.18),
같은 트랙·같은 차량으로만 주행하므로 좁은 도메인에 맞춰 파인튜닝하는 게 효율적이라고
판단함.

**진행 전 체크리스트**
1. Colab 런타임을 GPU로 설정 (런타임 > 런타임 유형 변경 > T4 이상)
2. Google Drive에 아래 구조로 데이터를 올려둘 것 (라벨링은 이 노트북이 대신 해주지
   않음 — 아래 '2. 데이터셋 라벨링' 섹션 참고):
   ```
   MyDrive/umk_twinlite/
     raw/
       images/*.jpg              # 라벨링 끝난 원본 프레임
       da_masks/*.png             # 같은 basename, 1채널, 0=배경 255=주행가능영역
       ll_masks/*.png             # 같은 basename, 1채널, 0=배경 255=차선(노란+흰색 구분 없음)
     pretrained/                  # (선택) 미리 받아둔 공식 pretrained .pth
   ```
3. **주의**: 이전 대화에서 분석한 화면 녹화(`.mov`, NoMachine 원격 데스크톱 캡처)는
   디버그 창(텍스트/박스/컬러 오버레이)이 카메라 화면 위에 그대로 찍혀 있어 학습
   이미지로 못 씀 — 라벨링용 원본 프레임은 카메라에서 직접 뽑거나, 실차에서 raw
   프레임을 따로 저장하는 기능을 추가해서 모아야 함.
4. **이번 배치(2026-08-10) 처리 이력**: 실차에서 뽑은 원본 프레임 2123장을 로컬(Mac)에서
   먼저 정리함 — ①기존 `best.onnx`로 da/ll 커버리지 지표를 계산해 실패/성공 자동 triage,
   ②`track_drive`가 쓰는 ROI(`DL_ROI_Y0`~`Y1`)를 좌/중앙/우 3구간으로 나눠 차선 미검출
   패턴 확인, ③라바콘 검출기(`cone_best.pt`, YOLOv8)로 라바콘 프레임이 30개 통과
   구간·542장에 걸쳐 과대표집된 걸 확인해 구간별 stride+최고신뢰도 프레임 보존 방식으로
   143장까지 축소, ④perceptual hash(dHash)로 정지 반복 촬영/같은 지점 재방문 등 근접
   중복 프레임을 제거하고 "3선 다 잡힌 쉬운 성공" 프레임도 추가로 솎아냄(콘 비중이
   상대적으로 부풀어서 한 번 더 축소) → 최종 **470장**으로 다이어트. 초안(draft) da/ll
   마스크는 이미 만들어둔 오버레이 이미지 색상에서 역산해서(onnx 재추론 없이) 로컬에서
   생성 완료, `raw/images_todo`/`raw/da_masks_draft`/`raw/ll_masks_draft`로 Drive에
   이미 업로드해둔 상태 — 아래 '2. 데이터셋 라벨링' 섹션은 이 470장에 대해서는 3단계
   (사람 보정)부터 시작하면 됨.

## 0. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/umk_twinlite'  # 필요하면 경로 수정
WORK_DIR = '/content/work'
REPO_DIR = f'{WORK_DIR}/TwinLiteNetPlus'
DATA_DIR = f'{WORK_DIR}/bdd100k'   # BDD100K.py가 '../bdd100k'로 하드코딩 참조하는 경로 (레포 이름 그대로 둬야 함)

import os
os.makedirs(WORK_DIR, exist_ok=True)
print('OK')

In [ ]:
!nvidia-smi -L  # GPU 배정 확인 (안 나오면 런타임 유형을 GPU로 바꿀 것)

In [ ]:
%cd {WORK_DIR}
!git clone --depth 1 https://github.com/chequanghuy/TwinLiteNetPlus.git
%cd {REPO_DIR}

`requirements.txt`는 `torch==1.8.0`처럼 오래된 버전을 지정하는데, 이 버전은 지금
Colab 환경(최신 CUDA)에서 설치가 잘 안 될 수 있어 **torch/torchvision은 Colab에
이미 깔린 버전을 그대로 쓰고**, 나머지 의존성만 버전 고정 없이 설치한다(대부분
opencv/albumentations 최신 버전과 호환됨 — 만약 특정 API에서 에러 나면 그때
버전을 좁혀서 재설치).

In [ ]:
!pip install -q albumentations opencv-python-headless scikit-learn scipy pillow tqdm timm matplotlib pyyaml gdown onnx onnxruntime
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())

## 1. Pretrained 가중치 받기

공식 저장소 README의 pretrained 폴더(구글 드라이브)에서 받는다. 폴더 전체 다운로드가
gdown으로 실패하면, 브라우저에서 직접 받아 `pretrained/{config}.pth`로 업로드할 것
— 링크: https://drive.google.com/drive/folders/1EqBzUw0b17aEumZmWYrGZmbx_XJqU-vz

`CONFIG`는 `nano/small/medium/large` 중 하나 — Jetson 실시간성을 이미 신경쓰고
있는 프로젝트라 **`small`을 기본값으로 권장**(속도/정확도 균형), 여유 있으면
`medium`/`large`로 A/B 비교해볼 것.

In [ ]:
CONFIG = 'medium'  # 'nano' | 'small' | 'medium' | 'large' — small(8~128ch)에서 medium(16~256ch)으로 한 단계 상향, ll처럼 얇고 복잡한 구조를 더 잘 표현하길 기대

import os
os.makedirs(f'{REPO_DIR}/pretrained', exist_ok=True)

drive_pretrained = f'{DRIVE_ROOT}/pretrained/{CONFIG}.pth'
local_pretrained = f'{REPO_DIR}/pretrained/{CONFIG}.pth'

if os.path.isfile(drive_pretrained):
    import shutil
    shutil.copy(drive_pretrained, local_pretrained)
    print('Drive에서 복사:', local_pretrained)
else:
    print('Drive에 없음 — gdown으로 폴더 전체 시도 (실패하면 수동 업로드 필요)')
    !gdown --folder 'https://drive.google.com/drive/folders/1EqBzUw0b17aEumZmWYrGZmbx_XJqU-vz' -O {REPO_DIR}/pretrained --remaining-ok

assert os.path.isfile(local_pretrained), f'{local_pretrained} 없음 — 수동으로 받아서 이 경로에 올려둘 것'
print('pretrained 준비 완료:', local_pretrained)

## 2. 데이터셋 라벨링 (이 노트북이 대신 못 해주는 부분)

라벨링(사람이 직접 확인/보정)은 자동화할 수 없음 — 대신 **지금 배포된 `best.onnx`
(원조 TwinLiteNet)의 예측을 초안(draft) 마스크로 깔아서 보정 시간을 줄이는 스크립트**를
제공한다. 순서:

1. 라벨링 안 된 원본 프레임을 `raw/images_todo/*.jpg`(또는 `.png`)로 Drive에 올린다.
2. 아래 셀로 기존 `best.onnx`를 돌려 `raw/da_masks_draft/`, `raw/ll_masks_draft/`에
   초안 마스크를 생성한다(임계값은 `track_drive/track_drive/config.py`의
   `DL_FG_THRESHOLD`(da, 0.5)/`DL_LL_FG_THRESHOLD`(ll, 0.7 — da보다 높음, BEV 원거리
   확률 blur 대응)와 동일하게 맞춤).
3. 로컬에서 (예: [labelme](https://github.com/wkentaro/labelme), GIMP, 또는 아무
   브러시 편집 도구) 초안 마스크를 열어 **틀린 부분만** 수정한다 — 특히 지금까지
   확인된 실패모드(노란선 미검출 구간, 글레어 구간)는 초안이 비어있을 테니 사람이
   직접 그려 넣어야 한다.
4. 보정 끝난 마스크를 `raw/images/`, `raw/da_masks/`, `raw/ll_masks/`로 옮긴다
   (파일명은 이미지와 마스크가 같은 basename이어야 함, 예: `f0001.jpg`/`f0001.png`).

**현재 상태(470장 배치)**: 1~2단계는 이미 로컬에서 끝내서 Drive에 올려둔 상태 —
`raw/images_todo/`(470장, `.png`), `raw/da_masks_draft/`, `raw/ll_masks_draft/`
전부 존재해야 함(아래 cell 실행하면 각각 몇 장인지 바로 확인됨). 이번 배치는
**3단계(사람 보정)부터** 진행하면 됨. 아래 cell-10은 이미 초안이 있는 파일은
재추론 없이 건너뛰도록 멱등하게 만들어놨음 — 나중에 새 프레임을 `images_todo/`에
추가로 올렸을 때만 그만큼만 새로 생성한다.

In [ ]:
# --- (선택) 기존 best.onnx로 초안 마스크 생성 — 이미 있는 초안은 건너뜀(멱등) ---
import onnxruntime as ort, cv2, numpy as np, glob, os

EXISTING_ONNX = f'{DRIVE_ROOT}/best.onnx'  # 지금 실차에 배포된 track_drive/track_drive/models/best.onnx를 올려둔 경로
TODO_DIR = f'{DRIVE_ROOT}/raw/images_todo'
DRAFT_DA_DIR = f'{DRIVE_ROOT}/raw/da_masks_draft'
DRAFT_LL_DIR = f'{DRIVE_ROOT}/raw/ll_masks_draft'
os.makedirs(DRAFT_DA_DIR, exist_ok=True)
os.makedirs(DRAFT_LL_DIR, exist_ok=True)

# 지금 몇 장이나 이미 준비돼 있는지 먼저 확인 — 470장 배치는 로컬(Mac)에서
# dataset_overlay 오버레이 색상으로 이미 역산해서 채워둔 상태라 재추론 없이 그대로 씀
for _sub, _path in [('images_todo', TODO_DIR), ('da_masks_draft', DRAFT_DA_DIR), ('ll_masks_draft', DRAFT_LL_DIR)]:
    _n = len(glob.glob(os.path.join(_path, '*.*'))) if os.path.isdir(_path) else 0
    print(f'{_sub}: {_n}장')

DA_THRESH = 0.5   # config.py DL_FG_THRESHOLD와 맞춤
LL_THRESH = 0.7   # config.py DL_LL_FG_THRESHOLD와 맞춤 — da보다 높음(BEV 원거리 확률 blur 대응, config.py 주석 참고)
MODEL_W, MODEL_H = 640, 360  # 지금 배포 중인 원조 TwinLiteNet 입력 크기 (config.py DL_INPUT_W/H)

def softmax_fg(logits_2ch):
    m = logits_2ch.max(axis=0, keepdims=True)
    e = np.exp(logits_2ch - m)
    return (e / e.sum(axis=0, keepdims=True))[1]

if os.path.isfile(EXISTING_ONNX):
    sess = ort.InferenceSession(EXISTING_ONNX, providers=['CPUExecutionProvider'])
    in_name = sess.get_inputs()[0].name
    out_names = [o.name for o in sess.get_outputs()]
    files = sorted(glob.glob(os.path.join(TODO_DIR, '*.jpg')) + glob.glob(os.path.join(TODO_DIR, '*.png')))

    # 이미 초안(da+ll 둘 다)이 있는 프레임은 재추론 없이 건너뜀 — 나중에 images_todo/에
    # 새 프레임만 추가로 올렸을 때 그만큼만 새로 생성하기 위함
    todo = []
    for fp in files:
        base = os.path.splitext(os.path.basename(fp))[0]
        has_da = os.path.isfile(os.path.join(DRAFT_DA_DIR, base + '.png'))
        has_ll = os.path.isfile(os.path.join(DRAFT_LL_DIR, base + '.png'))
        if not (has_da and has_ll):
            todo.append(fp)
    print(f'{len(files)}장 중 초안 없는 {len(todo)}장만 새로 생성')

    for fp in todo:
        bgr = cv2.imread(fp)
        h, w = bgr.shape[:2]
        resized = cv2.resize(bgr, (MODEL_W, MODEL_H))
        rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        blob = np.ascontiguousarray(np.transpose(rgb, (2, 0, 1))[None, ...])
        da_out, ll_out = sess.run(out_names, {in_name: blob})
        da_prob = cv2.resize(softmax_fg(da_out[0]), (w, h))
        ll_prob = cv2.resize(softmax_fg(ll_out[0]), (w, h))
        base = os.path.splitext(os.path.basename(fp))[0]
        cv2.imwrite(os.path.join(DRAFT_DA_DIR, base + '.png'), (da_prob >= DA_THRESH).astype(np.uint8) * 255)
        cv2.imwrite(os.path.join(DRAFT_LL_DIR, base + '.png'), (ll_prob >= LL_THRESH).astype(np.uint8) * 255)
    print('초안 생성 끝 —', DRAFT_DA_DIR, '/', DRAFT_LL_DIR, '를 내려받아 사람이 보정할 것')
else:
    print(f'{EXISTING_ONNX} 없음 — 이 단계는 건너뛰고 직접 라벨링해도 됨')

## 3. 라벨링된 데이터를 학습 폴더 구조로 정리 + train/val 분할

`raw/images/`, `raw/da_masks/`, `raw/ll_masks/`(2번 단계에서 사람이 보정 완료한 것)를
레포가 기대하는 `bdd100k/images|drivable_area_annotations|lane_line_annotations/{train,val}`
구조로 복사하고 분할한다.

In [ ]:
import os, glob, random, shutil, zipfile

# [5회차 갱신] pseudo_dataset -> pseudo_dataset_v2로 교체 (1068장: 사람라벨 134 +
# bootstrap_v2 모델 기반 자동생성 934, PROGRESS.md §2.12/§3 참고). da 소스가
# bootstrap_v2(134장 순수 사람 라벨로 학습, letterbox 버그/구 pseudo label 오염 없음)로
# 바뀌었고, 프레임 수도 470 -> 1068로 늘어남(dHash 근접중복 제거 상한까지 확장).
# 이 셀은 노트북의 "## 6-1 Stage 1" 셀들 뒤에 실행할 것 - 먼저 실행하면 Stage 1이
# 이 데이터로 DATA_DIR을 다시 덮어씀(의도된 동작).
DRIVE_ZIP = f'{DRIVE_ROOT}/pseudo_dataset_v2.zip'
DRIVE_PSEUDO = f'{DRIVE_ROOT}/pseudo_dataset_v2'
if not os.path.isdir(DRIVE_PSEUDO):
    assert os.path.isfile(DRIVE_ZIP), f'{DRIVE_ZIP} 없음 — Drive의 {DRIVE_ROOT}/ 에 로컬 pseudo_dataset_v2.zip을 올릴 것'
    with zipfile.ZipFile(DRIVE_ZIP) as zf:
        zf.extractall(DRIVE_ROOT)  # zip 안에 pseudo_dataset_v2/images|da_masks|ll_masks 구조로 들어있음
    print('압축 해제 완료:', DRIVE_PSEUDO)

SRC_IMG = f'{DRIVE_PSEUDO}/images'
SRC_DA = f'{DRIVE_PSEUDO}/da_masks'
SRC_LL = f'{DRIVE_PSEUDO}/ll_masks'
VAL_RATIO = 0.15
SEED = 42

names = sorted(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(SRC_IMG, '*.png')))
assert names, f'{SRC_IMG}에 png가 없음 — pseudo_dataset_v2 폴더 업로드 끝났는지 확인'
for n in names:
    for src, ext in [(SRC_DA, '.png'), (SRC_LL, '.png')]:
        assert os.path.isfile(os.path.join(src, n + ext)), f'{n}{ext} 마스크가 {src}에 없음'

random.Random(SEED).shuffle(names)
n_val = max(1, int(len(names) * VAL_RATIO))
val_names, train_names = names[:n_val], names[n_val:]
print(f'전체 {len(names)}장 -> train {len(train_names)} / val {len(val_names)}')

for split, split_names in [('train', train_names), ('val', val_names)]:
    for sub in ['images', 'drivable_area_annotations', 'lane_line_annotations']:
        os.makedirs(os.path.join(DATA_DIR, sub, split), exist_ok=True)
    for n in split_names:
        shutil.copy(os.path.join(SRC_IMG, n + '.png'), os.path.join(DATA_DIR, 'images', split, n + '.png'))
        shutil.copy(os.path.join(SRC_DA, n + '.png'), os.path.join(DATA_DIR, 'drivable_area_annotations', split, n + '.png'))
        shutil.copy(os.path.join(SRC_LL, n + '.png'), os.path.join(DATA_DIR, 'lane_line_annotations', split, n + '.png'))
print('정리 완료:', DATA_DIR)

## 4. 우리 환경 맞춤 증강 (오프라인) — 실제로 헷갈렸던 요소들을 노림

실차 영상/이번 세션에서 직접 확인된 혼동 요소들을 겨냥한 증강. **마스크는 바꾸지 않는다**
— 아래 효과들은 전부 차선의 물리적 위치를 바꾸지 않으므로 같은 라벨을 재사용해 이미지만
늘린다. `train` split에만 적용(val은 원본 분포를 유지해야 실제 성능 판단이 정확함).

- **글레어/노출과다**(`glare_aug`): 실차 영상 47초 지점 완전 미검출 사례(README §2.18) 겨냥.
  `RandomSunFlare` + 밝기/감마/색조.
- **모션 블러**(`motion_aug`): 이번 세션 몽타주 여러 장에서 실제로 관찰된 촬영 흔들림.
- **바닥 반사 스트릭**(`reflect_aug`): `frame_001082`처럼 계속 어려웠던, 광택 바닥에
  비친 빛 번짐. 글레어와 ROI/파라미터를 다르게 줘서 화면 아래쪽(바닥) 위주로 발생시킴.
- **가짜 얇은 선(distractor)**(`add_distractor_lines`): 체크무늬 타일 그라우트선처럼 차선과
  헷갈릴 수 있는 무관한 얇은 선을 라벨 없이(=배경으로) 무작위 추가 — 모델이 "얇고 긴 것 =
  차선"이 아니라 실제 라벨된 위치만 차선으로 구분하도록 하드 네거티브 역할.

매 증강 이미지마다 위 3가지 photometric 파이프라인 중 하나를 무작위로 고르고, 50% 확률로
distractor 선을 추가로 얹음 — `N_AUG_PER_IMAGE`장씩 원본마다 추가 생성.

In [ ]:
import albumentations as A
import cv2, os, glob, numpy as np, random

N_AUG_PER_IMAGE = 3  # 원본 1장당 증강 몇 장 추가할지 — 데이터가 적을수록 키울 것

glare_aug = A.Compose([
    A.OneOf([
        A.RandomSunFlare(flare_roi=(0, 0, 1, 0.6), src_radius=150, num_flare_circles_range=(3, 8), p=1.0),
        A.RandomBrightnessContrast(brightness_limit=(0.3, 0.6), contrast_limit=(-0.4, -0.1), p=1.0),
    ], p=0.7),
    A.RandomGamma(gamma_limit=(60, 160), p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=30, val_shift_limit=20, p=0.5),
])

motion_aug = A.Compose([
    A.MotionBlur(blur_limit=(5, 25), p=1.0),
])

reflect_aug = A.Compose([
    # 글레어(glare_aug)와 달리 화면 아래쪽(바닥) 위주로 반사 스트릭 발생 — frame_001082류 케이스 겨냥
    A.RandomSunFlare(flare_roi=(0.0, 0.4, 1.0, 1.0), src_radius=250, num_flare_circles_range=(2, 5), p=0.9),
    A.RandomBrightnessContrast(brightness_limit=(0.1, 0.3), contrast_limit=(-0.2, 0.1), p=0.5),
])

AUG_PIPELINES = [glare_aug, motion_aug, reflect_aug]


def add_distractor_lines(img, n_lines=(2, 5)):
    """체크무늬 타일 그라우트선 등 차선처럼 보일 수 있는 무관한 얇은 선을 무작위로 추가.
    마스크는 그대로 둠(=배경) -> 모델이 '얇고 긴 것'이 아니라 실제 라벨 위치만 차선으로
    학습하게 하는 하드 네거티브."""
    out = img.copy()
    h, w = out.shape[:2]
    for _ in range(random.randint(*n_lines)):
        x1, y1 = random.randint(0, w - 1), random.randint(int(h * 0.5), h - 1)
        length = random.randint(40, 200)
        angle = random.uniform(-60, 60)
        x2 = int(x1 + length * np.cos(np.radians(angle)))
        y2 = int(y1 - length * np.sin(np.radians(angle)))
        color = random.choice([(200, 200, 200), (230, 230, 210), (60, 60, 60)])
        thickness = random.randint(1, 3)
        overlay = out.copy()
        cv2.line(overlay, (x1, y1), (x2, y2), color, thickness, cv2.LINE_AA)
        alpha = random.uniform(0.25, 0.6)
        out = cv2.addWeighted(overlay, alpha, out, 1 - alpha, 0)
    return out


img_dir = os.path.join(DATA_DIR, 'images', 'train')
da_dir = os.path.join(DATA_DIR, 'drivable_area_annotations', 'train')
ll_dir = os.path.join(DATA_DIR, 'lane_line_annotations', 'train')

# [멱등 처리] 이 셀을 재실행하면 이전에 만든 '_aug*' 파일까지 '원본'으로 착각해서 또
# 증강해버려 기하급수적으로 불어나는 문제가 있었음(원본 400장 -> 5200장까지 폭증 사례).
# 매번 시작할 때 이전 증강 결과부터 지우고 순수 원본만 남긴 뒤 다시 생성.
n_removed = 0
for d in (img_dir, da_dir, ll_dir):
    for p in glob.glob(os.path.join(d, '*_aug*.png')):
        os.remove(p)
        n_removed += 1
if n_removed:
    print(f'이전 증강 파일 {n_removed}개 정리함 (재실행 대비)')

orig_names = sorted(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(img_dir, '*.png')))
print(f'원본 train {len(orig_names)}장에 각 {N_AUG_PER_IMAGE}장씩 증강 생성')

for n in orig_names:
    img = cv2.imread(os.path.join(img_dir, n + '.png'))
    for k in range(N_AUG_PER_IMAGE):
        pipeline = random.choice(AUG_PIPELINES)
        aug_img = pipeline(image=img)['image']  # 마스크는 증강 대상 아님 -> 그대로 재사용
        if random.random() < 0.5:
            aug_img = add_distractor_lines(aug_img)
        out_name = f'{n}_aug{k}'
        cv2.imwrite(os.path.join(img_dir, out_name + '.png'), aug_img)
        shutil_src_da = os.path.join(da_dir, n + '.png')
        shutil_src_ll = os.path.join(ll_dir, n + '.png')
        cv2.imwrite(os.path.join(da_dir, out_name + '.png'), cv2.imread(shutil_src_da, 0))
        cv2.imwrite(os.path.join(ll_dir, out_name + '.png'), cv2.imread(shutil_src_ll, 0))

final_count = len(glob.glob(os.path.join(img_dir, '*.png')))
print(f'증강 후 train 총 {final_count}장 (원본 {len(orig_names)}장 -> {final_count}장)')

In [ ]:
# 증강 결과 눈으로 확인 (원본 vs 증강 몇 장 비교)
import matplotlib.pyplot as plt

sample_names = orig_names[:3]
fig, axes = plt.subplots(len(sample_names), N_AUG_PER_IMAGE + 1, figsize=(4 * (N_AUG_PER_IMAGE + 1), 4 * len(sample_names)))
for row, n in enumerate(sample_names):
    orig = cv2.cvtColor(cv2.imread(os.path.join(img_dir, n + '.png')), cv2.COLOR_BGR2RGB)
    axes[row, 0].imshow(orig); axes[row, 0].set_title(f'{n} (원본)'); axes[row, 0].axis('off')
    for k in range(N_AUG_PER_IMAGE):
        aug = cv2.cvtColor(cv2.imread(os.path.join(img_dir, f'{n}_aug{k}.png')), cv2.COLOR_BGR2RGB)
        axes[row, k + 1].imshow(aug); axes[row, k + 1].set_title(f'aug{k}'); axes[row, k + 1].axis('off')
plt.tight_layout(); plt.show()

## 5. 파인튜닝용 hyperparameter yaml 준비

기본 `hyperparameters/twinlitev2_hyper.yaml`을 복사해서 두 가지만 바꾼다:
- `lr`을 낮춤(스크래치 학습이 아니라 파인튜닝이므로 10배 낮게 시작)
- `prob_crop`을 0으로 — 기본 `width_crop=960/height_crop=540`이 우리 캡처
  해상도보다 클 수 있어 `RandomCrop`이 에러날 위험이 있음(트랙 이미지 실제
  해상도를 확인했다면 `width_crop`/`height_crop`을 그에 맞게 다시 켜도 됨).

In [ ]:
import yaml

with open(f'{REPO_DIR}/hyperparameters/twinlitev2_hyper.yaml') as f:
    hyp = yaml.safe_load(f)

hyp['lr'] = hyp['lr'] * 0.1
hyp['prob_crop'] = 0.0
# da(mIoU 0.93까지 금방 수렴)에 비해 ll은 훨씬 어려운데, loss.py의 TotalLoss가 원래
# da/ll 손실을 1:1로 그냥 더해서 ll이 상대적으로 덜 밀어붙여짐 -> ll 쪽에 가중치를 더 줌
hyp['ll_loss_weight'] = 2.0

FINETUNE_HYP = f'{REPO_DIR}/hyperparameters/finetune_hyper.yaml'
with open(FINETUNE_HYP, 'w') as f:
    yaml.safe_dump(hyp, f)
print('저장:', FINETUNE_HYP, '| lr =', hyp['lr'], '| prob_crop =', hyp['prob_crop'],
      '| ll_loss_weight =', hyp['ll_loss_weight'])

# --- loss.py 패치: TotalLoss가 da/ll을 1:1로 더하던 걸 ll_loss_weight만큼 ll 쪽에 더 주게 수정 ---
# (멱등 처리: 이 셀을 레포 재클론 없이 두 번 실행해도 안전하게 - 이미 패치돼 있으면 스킵)
loss_path = f'{REPO_DIR}/loss.py'
with open(loss_path) as f:
    loss_src = f.read()

if 'self.ll_weight' in loss_src:
    print('loss.py 이미 패치돼 있음 -> 스킵')
else:
    old_init = '''        self.seg_tver_da = TverskyLoss(mode="multiclass", alpha=alpha1, beta=1-alpha1, gamma=gamma1, from_logits=True)
        self.seg_tver_ll = TverskyLoss(mode="multiclass", alpha=alpha2, beta=1-alpha2, gamma=gamma2, from_logits=True)
        self.seg_focal = FocalLossSeg(mode="multiclass", alpha=alpha3, gamma=gamma3)'''
    new_init = '''        self.seg_tver_da = TverskyLoss(mode="multiclass", alpha=alpha1, beta=1-alpha1, gamma=gamma1, from_logits=True)
        self.seg_tver_ll = TverskyLoss(mode="multiclass", alpha=alpha2, beta=1-alpha2, gamma=gamma2, from_logits=True)
        self.seg_focal = FocalLossSeg(mode="multiclass", alpha=alpha3, gamma=gamma3)
        self.ll_weight = hyp.get("ll_loss_weight", 1.0)  # [finetune 추가] ll을 더 세게 밀어붙임'''

    old_sum = '''        tversky_loss,focal_loss=tversky_da_loss+tversky_ll_loss,focal_da_loss+ focal_ll_loss'''
    new_sum = '''        tversky_loss,focal_loss=tversky_da_loss+self.ll_weight*tversky_ll_loss,focal_da_loss+self.ll_weight*focal_ll_loss'''

    assert old_init in loss_src, 'loss.py의 TotalLoss.__init__ 코드가 예상과 달라서 패치 실패 — 레포 버전 확인 필요'
    assert old_sum in loss_src, 'loss.py의 TotalLoss.forward 코드가 예상과 달라서 패치 실패'
    loss_src = loss_src.replace(old_init, new_init).replace(old_sum, new_sum)

    with open(loss_path, 'w') as f:
        f.write(loss_src)
    print(f'loss.py 패치 완료 -> TotalLoss가 이제 ll 손실에 {hyp["ll_loss_weight"]}배 가중치를 줌')

# --- utils.py 패치: poly_lr_scheduler가 매 에폭 모든 param group의 lr을 동일하게 덮어쓰던 걸,
# param group별 lr_mult를 반영하게 수정 (ll 디코더만 LR을 더 세게 주기 위한 사전 준비) ---
hyp['ll_lr_mult'] = 2.5  # ll 디코더(up_1_ll/up_2_ll/out_ll) LR = 기본 LR의 2.5배
with open(FINETUNE_HYP, 'w') as f:
    yaml.safe_dump(hyp, f)

# (아래 세 블록 전부 멱등 처리: 레포 재클론 없이 이 셀을 여러 번 실행해도 안전)
utils_path = f'{REPO_DIR}/utils.py'
with open(utils_path) as f:
    utils_src = f.read()

if "lr_mult" in utils_src and "[:,12:-12]" not in utils_src:
    print('utils.py 이미 패치돼 있음 -> 스킵')
else:
    old_sched = """def poly_lr_scheduler(args, hyp, optimizer, epoch, power=1.5):
    lr = round(hyp['lr'] * (1 - epoch / args.max_epochs) ** power, 8)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    return lr"""
    new_sched = """def poly_lr_scheduler(args, hyp, optimizer, epoch, power=1.5):
    lr = round(hyp['lr'] * (1 - epoch / args.max_epochs) ** power, 8)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr * param_group.get('lr_mult', 1.0)
    return lr"""
    if old_sched in utils_src:
        utils_src = utils_src.replace(old_sched, new_sched)

    # [finetune 추가, 중요] utils.py의 val()/val_one()이 da/ll 예측을 [:,12:-12]로 크롭하던 걸
    # 제거 - 원래 이 크롭은 BDD100K(720x1280) letterbox가 만드는 상하 12px 패딩을 없애기
    # 위한 것인데, 우리 이미지(640x480)는 letterbox 대신 plain resize로 바꿨으니(아래
    # BDD100K.py 패치) 이 크롭 자체가 불필요 + 오히려 실제 이미지 내용을 12px씩 잘라내는
    # 부작용이 있었음.
    utils_src = utils_src.replace("        da_predict = da_predict[:,12:-12]\n", "")
    utils_src = utils_src.replace("        ll_predict = ll_predict[:,12:-12]\n", "")
    utils_src = utils_src.replace("        predict = predict[:,12:-12]\n", "")  # val_one()용(안 쓰지만 일관성 위해 같이 수정

    with open(utils_path, 'w') as f:
        f.write(utils_src)
    print(f'utils.py 패치 완료 -> ll 디코더 LR {hyp["ll_lr_mult"]}배 + val() 크롭 제거')

# --- [finetune 추가, 핵심] BDD100K.py 패치: 우리 이미지(640x480, 4:3)가 letterbox를 거치면
# 좌우로 패딩되는데, 라벨은 그와 무관하게 그냥 640x360으로 눌러 리사이즈돼서 이미지/라벨
# 좌표가 서로 어긋나는 버그가 있었음(실차 dl_lane.py는 "letterbox 없이 plain resize"를
# 명시하고 있어서, 학습도 여기 맞춰야 함 - PROGRESS.md 참고). letterbox -> plain resize로
# 바꾸고, 라벨 리사이즈 높이도 360 -> 384(H_)로 통일해서 이미지/라벨을 동일하게 맞춤.
bdd_path = f'{REPO_DIR}/BDD100K.py'
with open(bdd_path) as f:
    bdd_src = f.read()

if 'letterbox(image' not in bdd_src:
    print('BDD100K.py 이미 패치돼 있음 -> 스킵')
else:
    bdd_src = bdd_src.replace('image = letterbox(image, (H_, W_))', 'image = cv2.resize(image, (W_, H_))')
    bdd_src = bdd_src.replace('cv2.resize(label, (W_, 360))', 'cv2.resize(label, (W_, H_))')
    bdd_src = bdd_src.replace('cv2.resize(label1, (W_, 360))', 'cv2.resize(label1, (W_, H_))')
    bdd_src = bdd_src.replace('cv2.resize(label2, (W_, 360))', 'cv2.resize(label2, (W_, H_))')
    with open(bdd_path, 'w') as f:
        f.write(bdd_src)
    print('BDD100K.py 패치 완료 -> letterbox 제거, plain resize(640x384)로 이미지/라벨 통일')

# --- loss.py 크롭도 제거(위에서 이미 ll 가중치 패치는 했으니 이어서) ---
with open(loss_path) as f:
    loss_src2 = f.read()
if '[:,:,12:-12]' not in loss_src2:
    print('loss.py 크롭 이미 제거돼 있음 -> 스킵')
else:
    loss_src2 = loss_src2.replace('out=outputs[:,:,12:-12]', 'out=outputs')
    loss_src2 = loss_src2.replace('out_da,out_ll=out_da[:,:,12:-12],out_ll[:,:,12:-12]', 'out_da,out_ll=out_da,out_ll')
    with open(loss_path, 'w') as f:
        f.write(loss_src2)
    print('loss.py 크롭 제거 완료')


## 6. 파인튜닝 스크립트 (`train.py` 기반 + pretrained 가중치 로드 추가)

공식 `train.py`엔 pretrained `.pth`(state_dict만 있는 배포용 가중치)를 초기화에
쓰는 옵션이 없다(`--resume`은 optimizer/epoch까지 포함된 `.tar` 체크포인트 전용).
`train.py`를 거의 그대로 복사하고 `--weight`로 pretrained state_dict를 학습 시작
전에 로드하는 부분만 추가한다.

In [ ]:
finetune_script = r'''
import os
import torch
import torch.optim.lr_scheduler
import torch.backends.cudnn as cudnn
import yaml
import math
from copy import deepcopy
from argparse import ArgumentParser

from model.model import TwinLiteNetPlus
from loss import TotalLoss
from utils import train, val, netParams, save_checkpoint, poly_lr_scheduler
import BDD100K

class ModelEMA:
    def __init__(self, model, decay=0.9999, updates=0):
        self.ema = deepcopy(model).eval()
        self.updates = updates
        self.decay = lambda x: decay * (1 - math.exp(-x / 2000))
        for p in self.ema.parameters():
            p.requires_grad_(False)

    def update(self, model):
        with torch.no_grad():
            self.updates += 1
            d = self.decay(self.updates)
            msd = model.state_dict()
            for k, v in self.ema.state_dict().items():
                if v.dtype.is_floating_point:
                    v *= d
                    v += (1. - d) * msd[k].detach()

def train_net(args, hyp):
    use_ema = args.ema
    cuda_available = torch.cuda.is_available()

    model = TwinLiteNetPlus(args)

    # --- [파인튜닝 추가분] pretrained state_dict 로드 (원조 train.py엔 없음) ---
    if args.weight and os.path.isfile(args.weight):
        state = torch.load(args.weight, map_location='cpu')
        if isinstance(state, dict) and 'state_dict' in state:
            state = state['state_dict']
        missing, unexpected = model.load_state_dict(state, strict=False)
        print(f'[finetune] pretrained 로드: {args.weight}')
        print(f'[finetune]   missing={len(missing)} unexpected={len(unexpected)}',
              '(대부분 0이어야 정상 — 많으면 --config가 pretrained와 다른 사이즈일 가능성)')
    else:
        print(f'[finetune] --weight 없음/파일 없음({args.weight}) — 랜덤 초기화로 진행')

    os.makedirs(args.savedir, exist_ok=True)

    trainLoader = torch.utils.data.DataLoader(
        BDD100K.Dataset(hyp, valid=False),
        batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers, pin_memory=True)

    valLoader = torch.utils.data.DataLoader(
        BDD100K.Dataset(hyp, valid=True),
        batch_size=args.batch_size, shuffle=False, num_workers=args.num_workers, pin_memory=True)

    if cuda_available:
        args.onGPU = True
        model = model.cuda()
        cudnn.benchmark = True

    print(f'Total network parameters: {netParams(model)}')

    criteria = TotalLoss(hyp)
    start_epoch = 0
    lr = hyp['lr']
    # [finetune 추가] ll 디코더(up_1_ll/up_2_ll/out_ll)만 LR을 hyp['ll_lr_mult']배로 줌 -
    # da는 이미 학습이 훨씬 쉬워서 공유 encoder LR을 그대로 올리면 da가 흔들릴 수 있음
    ll_lr_mult = hyp.get('ll_lr_mult', 1.0)
    ll_param_ids = set()
    ll_params = []
    for name, p in model.named_parameters():
        if '_ll' in name:
            ll_params.append(p)
            ll_param_ids.add(id(p))
    other_params = [p for p in model.parameters() if id(p) not in ll_param_ids]
    print(f'[finetune] ll 디코더 파라미터 {len(ll_params)}개 텐서 (LR x{ll_lr_mult}) / 나머지 {len(other_params)}개 텐서 (기본 LR)')
    optimizer = torch.optim.AdamW([
        {'params': other_params, 'lr': lr, 'lr_mult': 1.0},
        {'params': ll_params, 'lr': lr * ll_lr_mult, 'lr_mult': ll_lr_mult},
    ], betas=(hyp['momentum'], 0.999), eps=hyp['eps'], weight_decay=hyp['weight_decay'])

    ema = ModelEMA(model) if use_ema else None

    # [finetune 추가, 버그 수정] best_da_miou/best_ll_iou는 원래 아래에서 -1.0으로 매번
    # 초기화됐는데 checkpoint.pth.tar에 저장이 안 돼 있어서, --resume으로 이어서 돌 때마다
    # 리셋되고 resume 직후 첫 epoch이 실제 값과 무관하게 무조건 "새 best"로 찍혀 이전
    # best.pth를 더 나쁜 체크포인트로 덮어쓸 위험이 있었음. checkpoint에 저장/복원하도록 수정.
    best_da_miou = -1.0
    best_ll_iou = -1.0
    if args.resume and os.path.isfile(args.resume):
        if args.resume.endswith('.tar'):
            print(f"=> Loading checkpoint '{args.resume}'")
            checkpoint = torch.load(args.resume)
            start_epoch = checkpoint['epoch']
            model.load_state_dict(checkpoint['state_dict'])
            if use_ema:
                ema.ema.load_state_dict(checkpoint['ema_state_dict'])
                ema.updates = checkpoint['updates']
            optimizer.load_state_dict(checkpoint['optimizer'])
            best_da_miou = checkpoint.get('best_da_miou', -1.0)
            best_ll_iou = checkpoint.get('best_ll_iou', -1.0)
            print(f"=> Loaded checkpoint '{args.resume}' (epoch {checkpoint['epoch']}, "
                  f"best_da_miou={best_da_miou:.3f}, best_ll_iou={best_ll_iou:.3f})")
        else:
            print(f"=> No valid checkpoint found at '{args.resume}'")

    scaler = torch.cuda.amp.GradScaler()

    for epoch in range(start_epoch, args.max_epochs):
        model_file_name = os.path.join(args.savedir, f'model_{epoch}.pth')
        poly_lr_scheduler(args, hyp, optimizer, epoch)
        lr = optimizer.param_groups[0]['lr']
        ll_lr = optimizer.param_groups[1]['lr'] if len(optimizer.param_groups) > 1 else lr
        print(f'Learning rate: {lr} (ll decoder: {ll_lr})')

        model.train()
        ema = train(args, trainLoader, model, criteria, optimizer, epoch, scaler, args.verbose, ema if use_ema else None)

        model.eval()
        da_segment_results, ll_segment_results = val(valLoader, ema.ema if use_ema else model, args=args)

        print(f'Driving Area Segment: mIOU({da_segment_results[2]:.3f})')
        print(f'Lane Line Segment: Acc({ll_segment_results[0]:.3f}) IOU({ll_segment_results[1]:.3f})')

        torch.save(ema.ema.state_dict(), model_file_name) if use_ema else torch.save(model.state_dict(), model_file_name)
        if da_segment_results[2] > best_da_miou:
            best_da_miou = da_segment_results[2]
            best_path = os.path.join(args.savedir, 'best.pth')
            torch.save(ema.ema.state_dict() if use_ema else model.state_dict(), best_path)
            print(f'[finetune] 새 best(da) 저장: {best_path} (da mIoU={best_da_miou:.3f})')
        # [finetune 추가] best.pth는 da 기준이라 ll이 나빠져도 안 갱신됨 -> ll IOU 기준
        # best도 별도 파일로 저장해서, 나중에 da/ll 어느 쪽을 우선할지 골라 쓸 수 있게 함.
        if ll_segment_results[1] > best_ll_iou:
            best_ll_iou = ll_segment_results[1]
            best_ll_path = os.path.join(args.savedir, 'best_ll.pth')
            torch.save(ema.ema.state_dict() if use_ema else model.state_dict(), best_ll_path)
            print(f'[finetune] 새 best(ll) 저장: {best_ll_path} (ll IOU={best_ll_iou:.3f})')

        save_checkpoint({
            'epoch': epoch + 1,
            'state_dict': model.state_dict(),
            'ema_state_dict': ema.ema.state_dict() if use_ema else None,
            'updates': ema.updates if use_ema else None,
            'optimizer': optimizer.state_dict(),
            'lr': lr,
            'best_da_miou': best_da_miou,
            'best_ll_iou': best_ll_iou,
        }, os.path.join(args.savedir, 'checkpoint.pth.tar'))

        # [finetune 추가] 매 에폭마다 Drive에 백업 -> 세션 끊겨도 마지막 에폭까지는 안전.
        # Colab 로컬 디스크는 세션 종료 시 날아가는데, 원래 Drive 백업(노트북 cell 23)은
        # 학습이 다 끝난 뒤에만 실행돼서 중간에 끊기면 통째로 유실되는 문제를 막기 위함.
        if args.drive_backup_dir:
            import shutil as _shutil
            os.makedirs(args.drive_backup_dir, exist_ok=True)
            for _fn in ('checkpoint.pth.tar', 'best.pth', 'best_ll.pth'):
                _src = os.path.join(args.savedir, _fn)
                if os.path.isfile(_src):
                    _shutil.copy(_src, os.path.join(args.drive_backup_dir, _fn))
            print(f'[finetune] epoch {epoch} 체크포인트 Drive 백업 완료: {args.drive_backup_dir}')

if __name__ == '__main__':
    parser = ArgumentParser()
    parser.add_argument('--max_epochs', type=int, default=100)
    parser.add_argument('--num_workers', type=int, default=4)
    parser.add_argument('--batch_size', type=int, default=8)
    parser.add_argument('--savedir', default='./finetune_out')
    parser.add_argument('--hyp', type=str, default='./hyperparameters/finetune_hyper.yaml')
    parser.add_argument('--resume', type=str, default='')
    parser.add_argument('--weight', type=str, default='', help='파인튜닝 시작점 pretrained state_dict(.pth)')
    parser.add_argument('--config', default='small')
    parser.add_argument('--verbose', action='store_true')
    parser.add_argument('--ema', action='store_true')
    parser.add_argument('--drive_backup_dir', type=str, default='', help='매 에폭마다 체크포인트를 이 경로(Drive)에 백업 - 비우면 백업 안 함')
    args = parser.parse_args()

    with open(args.hyp, errors='ignore') as f:
        hyp = yaml.safe_load(f)

    train_net(args, hyp.copy())
'''

with open(f'{REPO_DIR}/finetune.py', 'w') as f:
    f.write(finetune_script)
print('작성 완료:', f'{REPO_DIR}/finetune.py')

## 6-1. [5회차 추가] Stage 1: bootstrap_v2(134장)로 먼저 미니 파인튜닝

**실행 순서 주의**: 노트북 순서상 cell-12(470장 pseudo_dataset 준비)가 이 섹션보다
먼저 나오지만, **지금은 건너뛸 것** — cell-12/13/14/15(470장 데이터 준비+증강)는
전부 스킵하고, **0~11번(환경설정~pretrained) → 16~19번(hyp/패치/finetune.py 작성)
→ 바로 아래 Stage 1 셀 3개** 순서로 실행할 것. (cell-12를 실수로 먼저 돌려도 무해함
— 아래 Stage 1 데이터 준비 셀이 `DATA_DIR`을 다시 지우고 채우므로 덮어써짐.)

PROGRESS.md §2.12에서 발견한 문제: 지금까지 470장 학습에 쓴 da pseudo label(396장)이
letterbox 버그로 학습된 구모델이 만든 거라 커브 구간에서 도로 폭을 과다포함하는
편향을 물려받음. 이 편향을 끊으려면, **pseudo label을 만드는 데 쓸 모델부터
편향 없는 사람 라벨만으로 다시 학습**해야 함.

`bootstrap_v2.zip`(기존 74장 사람 da+ll + 신규 60장 사람 da만/ll은 YOLOPv2로 채움 =
134장, 로컬 `scripts/build_bootstrap_v2.py`로 생성) 먼저 Drive
(`{DRIVE_ROOT}/bootstrap_v2.zip`)에 올려둘 것. 아래 3개 셀로 이 134장만 먼저
미니 파인튜닝 -> 나온 모델을 로컬로 받아서 `build_pseudo_label_dataset.py`의
`DA_ONNX`로 교체해 나머지(원래 470장 중 자동생성 396장)의 da pseudo label을
재생성 -> 로컬에서 `pseudo_dataset`(v2)을 다시 zip으로 묶어 Drive에 재업로드 ->
**그 다음에야 cell-12(470장+ 학습 데이터 준비)로 돌아가서 이어서 실행**하면 됨
(cell-12가 `DATA_DIR`을 pseudo_dataset 기준으로 다시 덮어써서 정상 동작).

In [ ]:
import os, glob, random, shutil, zipfile

# [Stage 1 전용 pretrained 확보] CONFIG 변수가 'medium' 등 다른 값이어도 Stage 1은
# 항상 small을 쓰므로, cell-8과 무관하게 여기서 직접 pretrained/small.pth를 확보.
_small_pretrained = f'{REPO_DIR}/pretrained/small.pth'
if not os.path.isfile(_small_pretrained):
    _drive_small = f'{DRIVE_ROOT}/pretrained/small.pth'
    if os.path.isfile(_drive_small):
        shutil.copy(_drive_small, _small_pretrained)
        print('Drive에서 small.pth 복사:', _small_pretrained)
    else:
        print('Drive에 small.pth 없음 -> gdown으로 폴더 전체 시도')
        os.makedirs(f'{REPO_DIR}/pretrained', exist_ok=True)
        get_ipython().system("gdown --folder 'https://drive.google.com/drive/folders/1EqBzUw0b17aEumZmWYrGZmbx_XJqU-vz' -O {REPO_DIR}/pretrained --remaining-ok")
assert os.path.isfile(_small_pretrained), f'{_small_pretrained} 없음 - 수동으로 받아서 이 경로에 올려둘 것'
print('Stage 1 pretrained 준비 완료:', _small_pretrained)

# Stage 1 데이터 준비 - bootstrap_v2(134장)를 bdd100k/ 구조로 정리.
# 주의: cell-12(pseudo_dataset_v2 준비)와 같은 DATA_DIR을 재사용함(BDD100K.py가
# '../bdd100k' 경로를 하드코딩 참조하기 때문, cell-2 주석 참고) - 이 셀 이후 Stage 1
# 학습까지 마치고, 나중에 cell-12를 실행하면 pseudo_dataset_v2로 덮어써짐(정상).
DRIVE_ZIP_V2 = f'{DRIVE_ROOT}/bootstrap_v2.zip'
DRIVE_BOOTSTRAP_V2 = f'{DRIVE_ROOT}/bootstrap_v2'
if not os.path.isdir(DRIVE_BOOTSTRAP_V2):
    assert os.path.isfile(DRIVE_ZIP_V2), f'{DRIVE_ZIP_V2} 없음 - Drive의 {DRIVE_ROOT}/ 에 로컬 bootstrap_v2.zip을 올릴 것'
    with zipfile.ZipFile(DRIVE_ZIP_V2) as zf:
        zf.extractall(DRIVE_ROOT)
    print('압축 해제 완료:', DRIVE_BOOTSTRAP_V2)

SRC_IMG_V2 = f'{DRIVE_BOOTSTRAP_V2}/images'
SRC_DA_V2 = f'{DRIVE_BOOTSTRAP_V2}/da_masks'
SRC_LL_V2 = f'{DRIVE_BOOTSTRAP_V2}/ll_masks'
VAL_RATIO_V2 = 0.15
SEED_V2 = 42

# 기존 bdd100k/ 내용을 지우고 bootstrap_v2로 새로 채움(재실행 대비 멱등 - cell-12를
# 먼저 실행했어도 여기서 덮어쓰므로 문제없음)
if os.path.isdir(DATA_DIR):
    shutil.rmtree(DATA_DIR)

names_v2 = sorted(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(SRC_IMG_V2, '*.png')))
assert names_v2, f'{SRC_IMG_V2}에 png가 없음'
random.Random(SEED_V2).shuffle(names_v2)
n_val_v2 = max(1, int(len(names_v2) * VAL_RATIO_V2))
val_names_v2, train_names_v2 = names_v2[:n_val_v2], names_v2[n_val_v2:]
print(f'bootstrap_v2 전체 {len(names_v2)}장 -> train {len(train_names_v2)} / val {len(val_names_v2)}')

for split, split_names in [('train', train_names_v2), ('val', val_names_v2)]:
    for sub in ['images', 'drivable_area_annotations', 'lane_line_annotations']:
        os.makedirs(os.path.join(DATA_DIR, sub, split), exist_ok=True)
    for n in split_names:
        shutil.copy(os.path.join(SRC_IMG_V2, n + '.png'), os.path.join(DATA_DIR, 'images', split, n + '.png'))
        shutil.copy(os.path.join(SRC_DA_V2, n + '.png'), os.path.join(DATA_DIR, 'drivable_area_annotations', split, n + '.png'))
        shutil.copy(os.path.join(SRC_LL_V2, n + '.png'), os.path.join(DATA_DIR, 'lane_line_annotations', split, n + '.png'))
print('Stage 1 데이터 정리 완료:', DATA_DIR)

In [ ]:
%cd {REPO_DIR}
MAX_EPOCHS_V2 = 40
BATCH_SIZE_V2 = 8

# Stage 1 학습 - 134장(순수 사람 da 기반)만으로 small 미니 파인튜닝.
# savedir을 470장 학습(finetune_out_470)과 분리해서 서로 안 건드림.
_resume_ckpt_v2 = f'{DRIVE_ROOT}/finetune_out_bootstrap_v2_live/checkpoint.pth.tar'
RESUME_FLAG_V2 = f'--resume {_resume_ckpt_v2}' if os.path.isfile(_resume_ckpt_v2) else ''
if RESUME_FLAG_V2:
    print(f'[재개] Drive 백업 체크포인트 발견 -> 이어서 학습: {_resume_ckpt_v2}')
else:
    print('[신규] Drive 백업 체크포인트 없음 -> 처음부터 학습')

!python finetune.py \
    --config small \
    --weight pretrained/small.pth \
    --hyp hyperparameters/finetune_hyper.yaml \
    --max_epochs {MAX_EPOCHS_V2} \
    --batch_size {BATCH_SIZE_V2} \
    --savedir ./finetune_out_bootstrap_v2 \
    --drive_backup_dir {DRIVE_ROOT}/finetune_out_bootstrap_v2_live \
    {RESUME_FLAG_V2} \
    --ema --verbose

In [ ]:
import torch, os
from model.model import TwinLiteNetPlus
from argparse import Namespace

# Stage 1 모델(134장, small)을 onnx로 내보내기 - 이걸 로컬로 받아서
# build_pseudo_label_dataset.py의 DA_ONNX로 교체해서 396장 da pseudo label 재생성에 씀.
WEIGHT_PATH_V2 = f'{REPO_DIR}/finetune_out_bootstrap_v2/best.pth'
ONNX_OUT_V2 = f'{DRIVE_ROOT}/finetune_out_bootstrap_v2/twinlitenetplus_small_bootstrap_v2.onnx'
os.makedirs(os.path.dirname(ONNX_OUT_V2), exist_ok=True)

model_v2 = TwinLiteNetPlus(Namespace(config='small'))
model_v2.load_state_dict(torch.load(WEIGHT_PATH_V2, map_location='cpu'))
model_v2.eval()

dummy_v2 = torch.zeros(1, 3, 384, 640)
torch.onnx.export(
    model_v2, dummy_v2, ONNX_OUT_V2,
    input_names=['images'], output_names=['da', 'll'],
    dynamic_axes={'images': {0: 'batch'}, 'da': {0: 'batch'}, 'll': {0: 'batch'}},
    opset_version=12,
)
print('Stage 1 onnx 내보내기 완료:', ONNX_OUT_V2)
print('-> 이 파일(+있으면 .onnx.data)을 로컬 ~/Downloads/에 받아서 build_pseudo_label_dataset.py의 DA_ONNX 경로로 바꿀 것')

## 7. 파인튜닝 실행

`MAX_EPOCHS`/`BATCH_SIZE`는 데이터 양·GPU 메모리에 맞춰 조정. 데이터가 수백 장
수준이면 30~50 epoch, 배치는 T4 기준 `small`이면 8~16 정도로 시작해볼 것 —
val mIoU가 몇 epoch째 안 오르면 조기 종료해도 됨.

In [ ]:
%cd {REPO_DIR}
MAX_EPOCHS = 40
BATCH_SIZE = 8

# [5회차 갱신] 1068장(pseudo_dataset_v2) 학습 - savedir을 finetune_out_v2로 분리해서
# 기존 470장 결과(finetune_out_470)와 안 섞이게 함. pretrained에서 새로 시작.
# --drive_backup_dir: 매 에폭마다 Drive에 체크포인트 백업 -> 세션 끊겨도 마지막 에폭부터
# --resume으로 재개 가능(랩탑 닫고 나가야 할 때 등 대비, GPU 할당량 소진도 포함).
# 1068장은 470장 대비 약 2배 분량이라 epoch당 시간도 대략 2배 예상 - 할당량 소진에
# 더 유의할 것(PROGRESS.md §3).

# [자동 재개] GPU 할당량 소진/세션 끊김 등으로 다시 이 셀을 실행하는 경우, Drive에 백업된
# checkpoint.pth.tar가 있으면 자동으로 --resume에 그 경로를 넘겨서 이어서 학습한다.
# Drive는 이미 마운트돼 있으므로 로컬로 복사할 필요 없이 경로를 그대로 넘기면 됨.
_resume_ckpt = f'{DRIVE_ROOT}/finetune_out_v2_live/checkpoint.pth.tar'
RESUME_FLAG = f'--resume {_resume_ckpt}' if os.path.isfile(_resume_ckpt) else ''
if RESUME_FLAG:
    print(f'[재개] Drive 백업 체크포인트 발견 -> 이어서 학습: {_resume_ckpt}')
else:
    print('[신규] Drive 백업 체크포인트 없음 -> 처음부터 학습')

!python finetune.py \
    --config {CONFIG} \
    --weight pretrained/{CONFIG}.pth \
    --hyp hyperparameters/finetune_hyper.yaml \
    --max_epochs {MAX_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --savedir ./finetune_out_v2 \
    --drive_backup_dir {DRIVE_ROOT}/finetune_out_v2_live \
    {RESUME_FLAG} \
    --ema --verbose

학습 로그에서 epoch마다 찍히는 `Driving Area Segment: mIOU(...)` /
`Lane Line Segment: Acc(...) IOU(...)`가 값이 오르는지 확인. `finetune_out/best.pth`가
val da mIoU 기준 최고 체크포인트, **`finetune_out/best_ll.pth`가 val ll IOU 기준 최고
체크포인트**(둘이 다른 epoch일 수 있음 — da는 일찍 수렴하고 ll이 늦게/불안정하게 오르는
경향이 있어서, ll 성능이 더 중요하면 최종 export 시 `best_ll.pth`도 같이 비교해볼 것).

In [ ]:
import shutil, os
os.makedirs(f'{DRIVE_ROOT}/finetune_out_v2', exist_ok=True)
shutil.copytree(f'{REPO_DIR}/finetune_out_v2', f'{DRIVE_ROOT}/finetune_out_v2', dirs_exist_ok=True)
print('체크포인트를 Drive에 백업함:', f'{DRIVE_ROOT}/finetune_out_v2')

## 8. ONNX로 내보내기 (`track_drive`의 `TwinLiteNetEngine`과 그대로 호환되게)

현재 `track_drive/track_drive/perception/dl_lane.py`의 `DL_INPUT_NAME='images'`,
`DL_OUTPUT_NAMES=('da','ll')`에 맞춰 입출력 이름을 지정한다. 후처리(`softmax_fg`,
채널1=foreground)는 원조 TwinLiteNet과 동일한 2채널 로짓 구조라 코드 변경 없음.

**주의**: TwinLiteNet+의 학습 파이프라인(`BDD100K.py`)은 입력을 640×**384**로
쓴다(원조 TwinLiteNet은 640×**360**) — 이 모델로 교체하면
`track_drive/track_drive/config.py`의 `DL_INPUT_H = 360`을 **384**로 바꿔야 함
(다른 ROI 좌표들은 원본 프레임 절대좌표 기준이라 영향 없음 — `infer_raw()`가 항상
원본 크기로 다시 업샘플링해서 반환하기 때문).

In [ ]:
import torch
from model.model import TwinLiteNetPlus
from argparse import Namespace

WEIGHT_PATH = f'{REPO_DIR}/finetune_out_v2/best.pth'
ONNX_OUT = f'{DRIVE_ROOT}/finetune_out_v2/twinlitenetplus_{CONFIG}_finetuned_v2.onnx'
os.makedirs(os.path.dirname(ONNX_OUT), exist_ok=True)

model = TwinLiteNetPlus(Namespace(config=CONFIG))
model.load_state_dict(torch.load(WEIGHT_PATH, map_location='cpu'))
model.eval()

dummy = torch.zeros(1, 3, 384, 640)  # (N,C,H,W) — BDD100K.py 학습 해상도와 동일하게 맞춤
torch.onnx.export(
    model, dummy, ONNX_OUT,
    input_names=['images'], output_names=['da', 'll'],
    dynamic_axes={'images': {0: 'batch'}, 'da': {0: 'batch'}, 'll': {0: 'batch'}},
    opset_version=12,
)
print('내보내기 완료:', ONNX_OUT)

In [ ]:
# 내보낸 onnx가 실제로 pytorch와 같은 결과를 내는지 검증
import onnxruntime as ort, numpy as np

sess = ort.InferenceSession(ONNX_OUT, providers=['CPUExecutionProvider'])
x = dummy.numpy()
onnx_da, onnx_ll = sess.run(['da', 'll'], {'images': x})

with torch.no_grad():
    torch_da, torch_ll = model(dummy)

da_diff = np.abs(onnx_da - torch_da.numpy()).max()
ll_diff = np.abs(onnx_ll - torch_ll.numpy()).max()
print(f'da 최대 오차: {da_diff:.6f} | ll 최대 오차: {ll_diff:.6f} (1e-4 이하면 정상)')
assert da_diff < 1e-3 and ll_diff < 1e-3, 'ONNX 변환 결과가 PyTorch와 다름 — export 재확인 필요'
print('검증 통과')

## 9. 실차 반영

1. `twinlitenetplus_{config}_finetuned.onnx`를 다운로드해
   `track_drive/track_drive/models/`에 넣는다(파일명은 자유 — `best.onnx`를 직접
   덮어쓰거나 새 이름으로 두고 아래 경로만 바꿔도 됨).
2. `track_drive/track_drive/config.py`:
   - `DL_INPUT_H = 360` → **`384`**
   - 모델 경로를 새 파일로 바꾸려면 `perception/dl_lane.py`의
     `_default_model_path()`가 찾는 파일명(`models/best.onnx`)에 맞게 새 onnx를
     `best.onnx`로 이름 바꾸거나, `TwinLiteNetEngine(model_path=...)` 호출부에
     새 경로를 넘기도록 수정.
3. `python3 -m py_compile track_drive/track_drive/config.py
   track_drive/track_drive/perception/dl_lane.py`로 문법 확인(이 개발 환경엔
   ROS2/onnxruntime이 없어 실행 검증은 못 함 — 실제 추론 확인은 실차에서).
4. 실차에서 `DEBUG_VIZ_DL_LANE`으로 `ll_cov`/`ok_bands`/`branch:` 요약이
   개선됐는지, 특히 노란선 검출률과 글레어 구간 대응이 나아졌는지 확인.